In [ ]:
import requests
import glob
import logging
import os
import re
import json
from collections import defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime as dt
from time import sleep
from html.parser import HTMLParser

import scobility, scobility_scrape as scrape

In [ ]:
path_dst = scrape.setup_scrape(tourney='gs')

In [ ]:
p_landing_pages = os.path.join(path_dst, 'landing_pages')
if not os.path.exists(p_landing_pages):
    os.makedirs(p_landing_pages)

landing_page = {}
for style_index, style_link in enumerate([
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=1&typeid=1",
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=1&typeid=2",
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=2&typeid=1",
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=2&typeid=2",
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=3&typeid=1",
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=3&typeid=2",
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=4&typeid=1",
    r"https://groovestats.com/index.php?page=songscores&gameid=38&modeid=4&typeid=2",
]):
    p_landing = os.path.join(p_landing_pages, f'{style_index+1}.html')
    if not os.path.exists(p_landing):
        r = requests.get(style_link)
        with open(p_landing, 'w') as fp:
            fp.write(r.text)
    with open(p_landing, 'r') as fp:
        landing_page[style_index] = fp.read()


In [ ]:
class SonglistHTMLParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.in_songlist_table = False
        self.songlist_links = []

    def handle_starttag(self, tag, attrs):
        attr_map = {a[0]: a[1] for a in attrs}
        if tag == "table" and attr_map.get("id", "") == "ranking_scores":
            self.in_songlist_table = True
        if self.in_songlist_table and tag == "a":
            href_data = attr_map.get("href", "")
            if re.match(r"index.php\?page=songscores&gameid=\d+&songid=\d+&modeid=\d+&typeid=\d+&chartid=\d+",
                        href_data, re.I):
                self.songlist_links.append(href_data)

    def handle_endtag(self, tag):
        if tag == "table":
            self.in_songlist_table = False

def link_to_chart_id(link):
    link_match = re.match(r"index.php\?page=songscores&gameid=(\d+)&songid=(\d+)&modeid=(\d+)&typeid=(\d+)&chartid=(\d+)", link, re.I)
    if link_match is not None:
        # return int(link_match.group(5)) * 8 + \
        #        (int(link_match.group(4)) - 1) * 4 + \
        #        (int(link_match.group(3)) - 1)
        return int(link_match.group(5))
    else:
        return 0

songlist_links = []
chart_ids = []
for page_index, page_contents in landing_page.items():
    parser = SonglistHTMLParser()
    parser.feed(page_contents)
    songlist_links += parser.songlist_links
    for sl in songlist_links:
        print(sl)
chart_ids = [link_to_chart_id(link) for link in songlist_links]

print(chart_ids)
print(len(chart_ids))
print(len(set(chart_ids)))

In [ ]:
@dataclass
class GSScore:
    s_id: int = 0
    e_id: int = 0
    name: str = ""
    style: int = 0
    value: float = 0        # 100% -> 0.00, 0% -> 1.00

@dataclass
class GSChart:
    s_id: int = 0
    title: str = ""
    artist: str = ""
    slot: str = ""
    style: int = 0
    meter: int = 0
    
class ScoreHTMLParser(HTMLParser):
    def __init__(self, s_id=0):
        super().__init__()
        self.chart_info = GSChart(s_id=s_id)
        self.in_info_table = False
        self.in_info_row_name = False
        self.in_info_row_data = False
        self.row_name = ""
        self.in_scores_table = False
        self.score_in_progress = None
        self.in_profile_link = False
        self.in_score_cell = False
        self.score_cell_contents = ""
        self.scores = []

    def handle_starttag(self, tag, attrs):
        attr_map = {a[0]: a[1] for a in attrs}
        if tag == "table":
            if attr_map.get("id", "") == "ranking_scores":
                self.in_scores_table = True
            elif attr_map.get("id", "") == "ranking_options":
                self.in_info_table = True
        if self.in_scores_table:
            if tag == "tr":
                self.score_in_progress = GSScore(self.chart_info.s_id)
            elif tag == "a":
                href_data = attr_map.get("href", "")
                user_link = re.match(r"index.php\?page=profile&id=(\d+)&typeid=(\d+)", href_data, re.I)
                if user_link is not None:
                    self.score_in_progress.e_id = int(user_link.group(1))
                    self.score_in_progress.style = int(user_link.group(2))
                    self.in_profile_link = True
            elif tag == "td":
                if "scoreCell" in attr_map.get("class", ""):
                    self.in_score_cell = True
                    self.score_cell_contents = ""
        if self.in_info_table:
            if "ranking_side" in attr_map.get("class", ""):
                self.in_info_row_name = True
            elif "ranking_form" in attr_map.get("class", ""):
                self.in_info_row_data = True
            elif "ranking_head" in attr_map.get("class", ""):
                self.row_name = "Title:"
                self.in_info_row_data = True

    def handle_data(self, data):
        if self.in_profile_link:
            self.score_in_progress.name = data.strip()
        elif self.in_info_row_name:
            self.row_name = data
        elif self.in_info_row_data:
            if self.row_name == "Title:":
                self.chart_info.title = data
            elif self.row_name == "Artist:":
                self.chart_info.artist = data
            elif self.row_name == "Mode:":
                self.chart_info.slot = data
            elif self.row_name == "Type:":
                self.chart_info.style = ["", "Single", "Double"].index(data)
            elif self.row_name == "Difficulty:":
                self.chart_info.meter = int(data)
        elif self.in_score_cell:
            self.score_cell_contents += data

    def handle_endtag(self, tag):
        if tag == "table":
            self.in_scores_table = False
            self.in_info_table = False
        elif tag == "tr":
            if self.score_in_progress is not None:
                if self.score_in_progress.e_id > 0:
                    self.scores.append(self.score_in_progress)
            self.score_in_progress = None
        elif tag == "a":
            self.in_profile_link = False
        elif tag == "td":
            if self.in_score_cell:
                try: 
                    self.score_in_progress.value = 1.0 - float(self.score_cell_contents.strip()) / 100.0
                except Exception as e:
                    print(e)
            self.in_score_cell = False
            self.in_info_row_name = False
            self.in_info_row_data = False

def parseChartPage(link: str, path_dst: str):
    p_raw_charts = os.path.join(path_dst, 'raw_charts')
    if not os.path.exists(p_raw_charts):
        os.makedirs(p_raw_charts)

    s_id = link_to_chart_id(link)
    if s_id == 0:
        return {"chart_info": GSChart(), "scores": []}
    
    if not os.path.exists(os.path.join(p_raw_charts, f'{s_id}.html')):    
        r = requests.get(f'https://groovestats.com/' + link)
        with open(os.path.join(p_raw_charts, f'{s_id}.html'), 'w') as fp:
            fp.write(r.text)
    
    with open(os.path.join(p_raw_charts, f'{s_id}.html'), 'r') as fp:
        score_page = fp.read()

    parser = ScoreHTMLParser(s_id=s_id)
    parser.feed(score_page)
    return {"chart_info": parser.chart_info, "scores": parser.scores}

# chart_data = parseChartPage(r"index.php?page=songscores&gameid=38&songid=58&modeid=1&typeid=1&chartid=250")
# print(chart_data["chart_info"])
# for s in chart_data["scores"]:
#     print(s)

In [ ]:
charts = []
scores = []
for i, link in enumerate(songlist_links):
    sleep(1)
    try:
        chart_data = parseChartPage(link, path_dst)
        charts.append(chart_data["chart_info"])
        scores += chart_data["scores"]
    except Exception as e:
        print(f"!!! {i:4d} {link}")
        print(e)
    if i % 10 == 0:
        print(f"... {i:4d} charts parsed")

In [ ]:
# Charts
p_charts = os.path.join(path_dst, 'song_info')
if not os.path.exists(p_charts):
    os.makedirs(p_charts)
charts_scob = {}
for c in charts:
    chart_basis = scobility.Song().dump()
    chart_basis.update(
        s_id=c.s_id,
        hash=hash(c.s_id),
        title=c.title,
        artist=c.artist,
        meter=c.meter,
        slot=(c.slot == "Expert") and "Challenge" or c.slot,
        style=["", "Single", "Double"][c.style]
    )
    charts_scob[c.s_id] = chart_basis
with open(os.path.join(path_dst, 'charts.json'), 'w', encoding='utf-8') as fp:
    json.dump(charts_scob, fp)
for s_id, c in charts_scob.items():
    with open(os.path.join(p_charts, f'{s_id}.json'), 'w', encoding='utf-8') as fp:
        json.dump({"song": c}, fp)

# Scores
p_scores = os.path.join(path_dst, 'song_scores')
if not os.path.exists(p_scores):
    os.makedirs(p_scores)
scores_scob = defaultdict(list)
for s in scores:
    score_basis = scobility.Score().dump()
    score_basis.update(
        s_id=s.s_id,
        e_id=s.e_id,
        value=s.value
    )
    scores_scob[s.s_id].append(scobility.Score.load(score_basis))
for s_id, s_list in scores_scob.items():
    with open(os.path.join(p_scores, f'{s_id}.json'), 'w', encoding='utf-8') as fp:
        json.dump({"scores": [asdict(s) for s in s_list]}, fp)

# Players
p_entrants = os.path.join(path_dst, 'entrant_info')
if not os.path.exists(p_entrants):
    os.makedirs(p_entrants)
players = defaultdict(list)
for s in scores:
    # print(s)
    players[s.e_id].append(s)

players_scob = {}
for e_id, p in players.items():
    player_basis = scobility.Player().dump()
    player_basis.update(
        name=p[0].name,
        e_id=e_id,
        g_id=e_id
    )
    players_scob[e_id] = scobility.Player.load(player_basis)
for e_id, e_info in players_scob.items():
    with open(os.path.join(p_entrants, f'{e_id}.json'), 'w', encoding='utf-8') as fp:
        json.dump({"entrant": asdict(e_info), "charts": [asdict(c) for c in players[e_id]]}, fp)


In [ ]:
players